# Bronze para Silver

Limpeza básica e gravação das quatro tabelas Silver.

In [0]:
from pyspark.sql import functions as F

partidas_bronze = spark.table("workspace.bronze.campeonato_brasileiro_full")
gols_bronze = spark.table("workspace.bronze.campeonato_brasileiro_gols")
cartoes_bronze = spark.table("workspace.bronze.campeonato_brasileiro_cartoes")
estatisticas_bronze = spark.table("workspace.bronze.campeonato_brasileiro_estatisticas_full")

## Partidas

In [0]:
partidas = partidas_bronze.select(
    F.col("ID").cast("long").alias("partida_id"),
    F.col("rodata").cast("int").alias("rodada"),
    F.to_date("data", "dd/MM/yyyy").alias("data_partida"),
    F.trim("hora").alias("hora"),
    F.trim("mandante").alias("mandante"),
    F.trim("visitante").alias("visitante"),
    F.trim("formacao_mandante").alias("formacao_mandante"),
    F.trim("formacao_visitante").alias("formacao_visitante"),
    F.trim("tecnico_mandante").alias("tecnico_mandante"),
    F.trim("tecnico_visitante").alias("tecnico_visitante"),
    F.trim("vencedor").alias("vencedor"),
    F.trim(F.regexp_replace("arena", "\u00a0", " ")).alias("arena"),
    F.col("mandante_Placar").cast("int").alias("placar_mandante"),
    F.col("visitante_Placar").cast("int").alias("placar_visitante"),
    F.upper(F.trim("mandante_Estado")).alias("estado_mandante"),
    F.upper(F.trim("visitante_Estado")).alias("estado_visitante"),
    F.expr("try_cast(arrecadacao as double)").alias("arrecadacao")
).withColumn(
    "resultado",
    F.when(F.col("placar_mandante") > F.col("placar_visitante"), "mandante")
     .when(F.col("placar_mandante") < F.col("placar_visitante"), "visitante")
     .otherwise("empate")
).withColumn("versao_fonte", F.lit(18))  .withColumn("processado_em", F.current_timestamp())

display(partidas.limit(10))

## Gols e cartões

In [0]:
def adiciona_minutos(df):
    return (
        df.withColumn("minuto_original", F.trim("minuto"))
          .withColumn("minuto_base", F.expr("try_cast(regexp_extract(minuto_original, '^([0-9]+)', 1) as int)"))
          .withColumn("minuto_acrescimo", F.expr("try_cast(regexp_extract(minuto_original, '[+]([0-9]+)', 1) as int)"))
          .withColumn("minuto_ordem", F.col("minuto_base") + F.coalesce("minuto_acrescimo", F.lit(0)))
          .drop("minuto")
    )

gols = gols_bronze.select(
    F.col("partida_id").cast("long").alias("partida_id"),
    F.col("rodata").cast("int").alias("rodada"),
    F.trim("clube").alias("clube"),
    F.trim("atleta").alias("atleta"),
    F.col("minuto"),
    F.coalesce(F.trim("tipo_de_gol"), F.lit("Normal")).alias("tipo_gol")
)
gols = adiciona_minutos(gols).withColumn("versao_fonte", F.lit(18))     .withColumn("processado_em", F.current_timestamp())

cartoes = cartoes_bronze.select(
    F.col("partida_id").cast("long").alias("partida_id"),
    F.col("rodata").cast("int").alias("rodada"),
    F.trim("clube").alias("clube"),
    F.trim("cartao").alias("cartao"),
    F.trim("atleta").alias("atleta"),
    F.expr("try_cast(num_camisa as int)").alias("numero_camisa"),
    F.trim("posicao").alias("posicao"),
    F.col("minuto")
)
cartoes = adiciona_minutos(cartoes).withColumn("versao_fonte", F.lit(18))     .withColumn("processado_em", F.current_timestamp())

## Estatísticas

In [0]:
estatisticas = estatisticas_bronze.select(
    F.col("partida_id").cast("long").alias("partida_id"),
    F.col("rodata").cast("int").alias("rodada"),
    F.trim("clube").alias("clube"),
    F.col("chutes").cast("int").alias("chutes"),
    F.col("chutes_no_alvo").cast("int").alias("chutes_no_alvo"),
    F.expr("try_cast(regexp_replace(posse_de_bola, '%', '') as double)").alias("posse_bola_pct"),
    F.col("passes").cast("int").alias("passes"),
    F.expr("try_cast(regexp_replace(precisao_passes, '%', '') as double)").alias("precisao_passes_pct"),
    F.col("faltas").cast("int").alias("faltas"),
    F.col("cartao_amarelo").cast("int").alias("cartoes_amarelos"),
    F.col("cartao_vermelho").cast("int").alias("cartoes_vermelhos"),
    F.col("impedimentos").cast("int").alias("impedimentos"),
    F.col("escanteios").cast("int").alias("escanteios")
).withColumn(
    "estatisticas_disponiveis",
    F.col("posse_bola_pct").isNotNull()
).withColumn("versao_fonte", F.lit(18))  .withColumn("processado_em", F.current_timestamp())

display(estatisticas.filter("estatisticas_disponiveis").limit(10))

## Validações

In [0]:
duplicadas = partidas.groupBy("partida_id").count().filter("count > 1").count()
gols_orfaos = gols.join(partidas.select("partida_id"), "partida_id", "left_anti").count()
cartoes_orfaos = cartoes.join(partidas.select("partida_id"), "partida_id", "left_anti").count()
estatisticas_orfas = estatisticas.join(partidas.select("partida_id"), "partida_id", "left_anti").count()

validacao = spark.createDataFrame([
    ("IDs duplicados em partidas", duplicadas),
    ("Gols sem partida", gols_orfaos),
    ("Cartões sem partida", cartoes_orfaos),
    ("Estatísticas sem partida", estatisticas_orfas),
], ["teste", "ocorrencias"])

display(validacao)

if validacao.filter("ocorrencias > 0").count() > 0:
    raise ValueError("Falha nas validações")

## Gravação

In [0]:
tabelas = {
    "partidas": partidas,
    "gols": gols,
    "cartoes": cartoes,
    "estatisticas": estatisticas,
}

for nome, df in tabelas.items():
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"workspace.silver.{nome}"))

In [0]:
contagens = [(nome, df.count()) for nome, df in tabelas.items()]
display(spark.createDataFrame(contagens, ["tabela", "registros"]))
spark.sql("SHOW TABLES IN workspace.silver").show(truncate=False)